In [ ]:
import coremltools as ct
import numpy as np
from PIL import Image
from ultralytics import SAM
import cv2


IMAGE_PATH = "DJI_15-copy_jpg.rf.947a177207587775a6d0377b1173ca20.jpg"
MODEL_PATH = "yolo26m-krops-v0.1.mlpackage"

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45


# --------------------------------------------------
# IoU
# --------------------------------------------------

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter_area = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - inter_area

    return inter_area / union if union > 0 else 0


# --------------------------------------------------
# NMS
# --------------------------------------------------

def nms(boxes, scores, iou_threshold=0.45):
    order = np.argsort(scores)[::-1]

    selected = []

    while len(order) > 0:
        i = order[0]
        selected.append(i)

        remaining = []

        for j in order[1:]:
            if iou(boxes[i], boxes[j]) <= iou_threshold:
                remaining.append(j)

        order = np.array(remaining)

    return selected


# --------------------------------------------------
# Load Core ML
# --------------------------------------------------

model = ct.models.MLModel(MODEL_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")

# Your model expects 640x640
input_image = image.resize((640, 640))

prediction = model.predict({
    "image": input_image
})

# Find the output
print(prediction.keys())

raw = prediction["var_1515"]

print("Raw shape:", raw.shape)


# --------------------------------------------------
# Parse [1, 5, 8400]
# --------------------------------------------------

raw = np.asarray(raw)

# [1, 5, 8400] -> [8400, 5]
predictions = raw[0].T

boxes = []
scores = []

for pred in predictions:

    cx, cy, w, h, confidence = pred

    if confidence < CONF_THRESHOLD:
        continue

    # xywh -> xyxy
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2

    # Normalize to 0..1
    x1 /= 640
    y1 /= 640
    x2 /= 640
    y2 /= 640

    # Clamp
    x1 = max(0, min(1, x1))
    y1 = max(0, min(1, y1))
    x2 = max(0, min(1, x2))
    y2 = max(0, min(1, y2))

    boxes.append([x1, y1, x2, y2])
    scores.append(float(confidence))


# --------------------------------------------------
# NMS
# --------------------------------------------------

selected = nms(
    boxes,
    scores,
    IOU_THRESHOLD
)

boxes = [boxes[i] for i in selected]
scores = [scores[i] for i in selected]

print(f"Detected palms: {len(boxes)}")

# --------------------------------------------------
# Display Core ML detection result
# --------------------------------------------------

from PIL import ImageDraw, ImageFont

detector_image = image.copy()
draw = ImageDraw.Draw(detector_image)

original_width, original_height = detector_image.size

for box, score in zip(boxes, scores):

    x1, y1, x2, y2 = box

    # Normalized -> original image coordinates
    x1 = int(x1 * original_width)
    y1 = int(y1 * original_height)
    x2 = int(x2 * original_width)
    y2 = int(y2 * original_height)

    # Draw bounding box
    draw.rectangle(
        [x1, y1, x2, y2],
        outline="red",
        width=3
    )

    # Draw confidence
    text = f"Oil Palm {score:.2f}"

    draw.text(
        (x1, max(0, y1 - 20)),
        text,
        fill="red"
    )

print(f"Detected palms: {len(boxes)}")

display(detector_image)


for box, score in zip(boxes, scores):
    print(box, score)


# --------------------------------------------------
# Convert normalized boxes to 640x640 pixel boxes
# for SAM
# --------------------------------------------------

sam_boxes = []

for x1, y1, x2, y2 in boxes:

    sam_boxes.append([
        x1 * 640,
        y1 * 640,
        x2 * 640,
        y2 * 640
    ])


# --------------------------------------------------
# SAM
# --------------------------------------------------

sam = SAM("sam_l.pt")

results = sam(
    IMAGE_PATH,
    bboxes=sam_boxes
)

result = results[0]


# --------------------------------------------------
# Combine all SAM masks
# --------------------------------------------------

original = np.array(image.convert("RGBA"))

combined_mask = np.zeros(
    original.shape[:2],
    dtype=bool
)

for mask_tensor in result.masks.data:

    mask = mask_tensor.cpu().numpy()

    # SAM mask should correspond to original image resolution
    combined_mask |= mask > 0


# --------------------------------------------------
# Transparent background
# --------------------------------------------------

alpha = (combined_mask * 255).astype(np.uint8)

output = original.copy()
output[:, :, 3] = alpha

output_image = Image.fromarray(output)

display(output_image)

# output_image.save("oil_palm_segmented.png")

# print("Saved: oil_palm_segmented.png")

In [ ]:
import coremltools as ct
import numpy as np
from PIL import Image
from ultralytics import SAM
import cv2


IMAGE_PATH = "IMG_000117.jpg"
MODEL_PATH = "yolo26m-krops-v0.1.mlpackage"

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45


# --------------------------------------------------
# IoU
# --------------------------------------------------

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter_area = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - inter_area

    return inter_area / union if union > 0 else 0


# --------------------------------------------------
# NMS
# --------------------------------------------------

def nms(boxes, scores, iou_threshold=0.45):
    order = np.argsort(scores)[::-1]

    selected = []

    while len(order) > 0:
        i = order[0]
        selected.append(i)

        remaining = []

        for j in order[1:]:
            if iou(boxes[i], boxes[j]) <= iou_threshold:
                remaining.append(j)

        order = np.array(remaining)

    return selected


# --------------------------------------------------
# Load Core ML
# --------------------------------------------------

model = ct.models.MLModel(MODEL_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")

# Your model expects 640x640
input_image = image.resize((640, 640))

prediction = model.predict({
    "image": input_image
})

# Find the output
print(prediction.keys())

raw = prediction["var_1515"]

print("Raw shape:", raw.shape)


# --------------------------------------------------
# Parse [1, 5, 8400]
# --------------------------------------------------

raw = np.asarray(raw)

# [1, 5, 8400] -> [8400, 5]
predictions = raw[0].T

boxes = []
scores = []

for pred in predictions:

    cx, cy, w, h, confidence = pred

    if confidence < CONF_THRESHOLD:
        continue

    # xywh -> xyxy
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2

    # Normalize to 0..1
    x1 /= 640
    y1 /= 640
    x2 /= 640
    y2 /= 640

    # Clamp
    x1 = max(0, min(1, x1))
    y1 = max(0, min(1, y1))
    x2 = max(0, min(1, x2))
    y2 = max(0, min(1, y2))

    boxes.append([x1, y1, x2, y2])
    scores.append(float(confidence))


# --------------------------------------------------
# NMS
# --------------------------------------------------

selected = nms(
    boxes,
    scores,
    IOU_THRESHOLD
)

boxes = [boxes[i] for i in selected]
scores = [scores[i] for i in selected]

print(f"Detected palms: {len(boxes)}")

# --------------------------------------------------
# Display Core ML detection result
# --------------------------------------------------

from PIL import ImageDraw, ImageFont

detector_image = image.copy()
draw = ImageDraw.Draw(detector_image)

original_width, original_height = detector_image.size

for box, score in zip(boxes, scores):

    x1, y1, x2, y2 = box

    # Normalized -> original image coordinates
    x1 = int(x1 * original_width)
    y1 = int(y1 * original_height)
    x2 = int(x2 * original_width)
    y2 = int(y2 * original_height)

    # Draw bounding box
    draw.rectangle(
        [x1, y1, x2, y2],
        outline="red",
        width=3
    )

    # Draw confidence
    text = f"Oil Palm {score:.2f}"

    draw.text(
        (x1, max(0, y1 - 20)),
        text,
        fill="red"
    )

print(f"Detected palms: {len(boxes)}")

display(detector_image)


for box, score in zip(boxes, scores):
    print(box, score)


# --------------------------------------------------
# Convert normalized boxes to 640x640 pixel boxes
# for SAM
# --------------------------------------------------

original_width, original_height = image.size

sam_boxes = []

for x1, y1, x2, y2 in boxes:
    sam_boxes.append([
        x1 * original_width,
        y1 * original_height,
        x2 * original_width,
        y2 * original_height
    ])


# --------------------------------------------------
# SAM
# --------------------------------------------------

sam = SAM("sam_l.pt")

results = sam(
    IMAGE_PATH,
    bboxes=sam_boxes
)

result = results[0]


# --------------------------------------------------
# Combine all SAM masks
# --------------------------------------------------

original = np.array(image.convert("RGBA"))

combined_mask = np.zeros(
    original.shape[:2],
    dtype=bool
)

for mask_tensor in result.masks.data:

    mask = mask_tensor.cpu().numpy()

    # SAM mask should correspond to original image resolution
    combined_mask |= mask > 0


# --------------------------------------------------
# Transparent background
# --------------------------------------------------

alpha = (combined_mask * 255).astype(np.uint8)

output = original.copy()
output[:, :, 3] = alpha

output_image = Image.fromarray(output)

display(output_image)

# output_image.save("oil_palm_segmented.png")

# print("Saved: oil_palm_segmented.png")

In [ ]:
import coremltools as ct
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import SAM
import cv2
import os


# ============================================================
# CONFIG
# ============================================================

IMAGE_PATH = "IMG_000117.jpg"
MODEL_PATH = "yolo26m-krops-v0.1.mlpackage"

SAM_MODEL_PATH = "sam_l.pt"

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45

# Set True if you only want to test the highest-confidence
# detector box first.
TEST_ONLY_BEST_BOX = True

# Output files
DETECTOR_OUTPUT = "detector_boxes.png"
SAM_BOX_OUTPUT = "sam_input_boxes.png"
SAM_OUTPUT = "sam_segmentation.png"
OVERLAY_OUTPUT = "sam_overlay.png"


# ============================================================
# IOU
# ============================================================

def iou(box1, box2):
    """
    Calculate IoU between two boxes.

    Boxes are:
        [x1, y1, x2, y2]

    Coordinates are normalized to 0..1.
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)

    inter_area = inter_w * inter_h

    area1 = (
        max(0, box1[2] - box1[0])
        * max(0, box1[3] - box1[1])
    )

    area2 = (
        max(0, box2[2] - box2[0])
        * max(0, box2[3] - box2[1])
    )

    union = area1 + area2 - inter_area

    if union <= 0:
        return 0

    return inter_area / union


# ============================================================
# NMS
# ============================================================

def nms(boxes, scores, iou_threshold=0.45):
    """
    Standard Non-Maximum Suppression.

    Returns indices of boxes that survived NMS.
    """

    if len(boxes) == 0:
        return []

    order = np.argsort(scores)[::-1]

    selected = []

    while len(order) > 0:

        current = order[0]

        selected.append(current)

        remaining = []

        for index in order[1:]:

            overlap = iou(
                boxes[current],
                boxes[index]
            )

            if overlap <= iou_threshold:
                remaining.append(index)

        order = np.array(
            remaining,
            dtype=np.int64
        )

    return selected


# ============================================================
# LOAD IMAGE
# ============================================================

print("=" * 60)
print("Loading image...")
print("=" * 60)

image = Image.open(IMAGE_PATH).convert("RGB")

original_width, original_height = image.size

print("Original image size:")
print(
    f"  width  = {original_width}"
)
print(
    f"  height = {original_height}"
)


# ============================================================
# LOAD CORE ML MODEL
# ============================================================

print("\n" + "=" * 60)
print("Loading Core ML model...")
print("=" * 60)

model = ct.models.MLModel(
    MODEL_PATH
)

print("Core ML model loaded.")


# ============================================================
# PREPARE IMAGE FOR CORE ML
# ============================================================

print("\n" + "=" * 60)
print("Preparing image...")
print("=" * 60)

MODEL_SIZE = 640

input_image = image.resize(
    (MODEL_SIZE, MODEL_SIZE)
)

print(
    f"Model input size: "
    f"{MODEL_SIZE}x{MODEL_SIZE}"
)


# ============================================================
# CORE ML INFERENCE
# ============================================================

print("\n" + "=" * 60)
print("Running Core ML inference...")
print("=" * 60)

prediction = model.predict({
    "image": input_image
})

print(
    "Prediction keys:",
    prediction.keys()
)


# ============================================================
# GET RAW OUTPUT
# ============================================================

raw = prediction["var_1515"]

raw = np.asarray(raw)

print("\nRaw output shape:")
print(raw.shape)


# Expected:
#
# [1, 5, 8400]
#
# channel 0 = cx
# channel 1 = cy
# channel 2 = w
# channel 3 = h
# channel 4 = confidence


if raw.ndim != 3:
    raise RuntimeError(
        f"Unexpected output dimensions: {raw.shape}"
    )


if raw.shape[1] != 5:
    raise RuntimeError(
        "This script expects the Core ML model "
        f"to output [1, 5, N], but got {raw.shape}"
    )


# ============================================================
# TRANSPOSE
# ============================================================

#
# [1, 5, 8400]
#
# becomes
#
# [8400, 5]
#

predictions = raw[0].T

print(
    "Parsed predictions:",
    predictions.shape
)


# ============================================================
# DECODE DETECTIONS
# ============================================================

boxes = []
scores = []

for pred in predictions:

    cx = float(pred[0])
    cy = float(pred[1])
    w = float(pred[2])
    h = float(pred[3])

    confidence = float(pred[4])

    # ----------------------------------------
    # Confidence filtering
    # ----------------------------------------

    if confidence < CONF_THRESHOLD:
        continue

    # ----------------------------------------
    # xywh -> xyxy
    # ----------------------------------------

    x1 = cx - w / 2
    y1 = cy - h / 2

    x2 = cx + w / 2
    y2 = cy + h / 2

    # ----------------------------------------
    # Normalize from 640x640 -> 0..1
    # ----------------------------------------

    x1 /= MODEL_SIZE
    y1 /= MODEL_SIZE

    x2 /= MODEL_SIZE
    y2 /= MODEL_SIZE

    # ----------------------------------------
    # Clamp
    # ----------------------------------------

    x1 = max(0.0, min(1.0, x1))
    y1 = max(0.0, min(1.0, y1))

    x2 = max(0.0, min(1.0, x2))
    y2 = max(0.0, min(1.0, y2))

    # ----------------------------------------
    # Reject invalid boxes
    # ----------------------------------------

    if x2 <= x1:
        continue

    if y2 <= y1:
        continue

    boxes.append([
        x1,
        y1,
        x2,
        y2
    ])

    scores.append(
        confidence
    )


print(
    f"\nDetections before NMS: {len(boxes)}"
)


# ============================================================
# NMS
# ============================================================

selected = nms(
    boxes,
    scores,
    IOU_THRESHOLD
)

boxes = [
    boxes[i]
    for i in selected
]

scores = [
    scores[i]
    for i in selected
]


print(
    f"Detections after NMS: {len(boxes)}"
)


# ============================================================
# SORT BY CONFIDENCE
# ============================================================

combined = list(
    zip(boxes, scores)
)

combined.sort(
    key=lambda x: x[1],
    reverse=True
)

boxes = [
    item[0]
    for item in combined
]

scores = [
    item[1]
    for item in combined
]


# ============================================================
# PRINT DETECTIONS
# ============================================================

print("\n" + "=" * 60)
print("FINAL DETECTIONS")
print("=" * 60)

for i, (box, score) in enumerate(
    zip(boxes, scores)
):

    x1, y1, x2, y2 = box

    print(
        f"{i:02d} | "
        f"confidence={score:.4f} | "
        f"normalized="
        f"[{x1:.3f}, "
        f"{y1:.3f}, "
        f"{x2:.3f}, "
        f"{y2:.3f}]"
    )


# ============================================================
# DRAW DETECTOR BOXES
# ============================================================

print("\nCreating detector visualization...")

detector_image = image.copy()

draw = ImageDraw.Draw(
    detector_image
)

for i, (box, score) in enumerate(
    zip(boxes, scores)
):

    x1, y1, x2, y2 = box

    # normalized -> original image coordinates

    px1 = int(
        x1 * original_width
    )

    py1 = int(
        y1 * original_height
    )

    px2 = int(
        x2 * original_width
    )

    py2 = int(
        y2 * original_height
    )

    # Draw box

    draw.rectangle(
        [
            px1,
            py1,
            px2,
            py2
        ],
        outline="red",
        width=5
    )

    # Label

    label = (
        f"Palm {i} "
        f"{score:.2f}"
    )

    text_y = max(
        0,
        py1 - 25
    )

    draw.text(
        (
            px1,
            text_y
        ),
        label,
        fill="red"
    )


detector_image.save(
    DETECTOR_OUTPUT
)

print(
    f"Saved: {DETECTOR_OUTPUT}"
)

display(
    detector_image
)


# ============================================================
# CONVERT DETECTOR BOXES TO SAM BOXES
# ============================================================

sam_boxes = []

for box in boxes:

    x1, y1, x2, y2 = box

    sam_box = [
        x1 * original_width,
        y1 * original_height,
        x2 * original_width,
        y2 * original_height
    ]

    sam_boxes.append(
        sam_box
    )


# ============================================================
# OPTIONAL:
# ONLY TEST BEST DETECTION
# ============================================================

if TEST_ONLY_BEST_BOX:

    if len(sam_boxes) == 0:
        raise RuntimeError(
            "No detector boxes available."
        )

    sam_boxes = [
        sam_boxes[0]
    ]

    print(
        "\nTEST_ONLY_BEST_BOX enabled."
    )

    print(
        "Using only highest-confidence box:"
    )

    print(
        sam_boxes[0]
    )


# ============================================================
# DRAW EXACT BOXES SENT TO SAM
# ============================================================

print("\nCreating SAM input visualization...")

sam_debug = image.copy()

draw = ImageDraw.Draw(
    sam_debug
)

for i, box in enumerate(
    sam_boxes
):

    x1, y1, x2, y2 = [
        int(v)
        for v in box
    ]

    draw.rectangle(
        [
            x1,
            y1,
            x2,
            y2
        ],
        outline="blue",
        width=6
    )

    draw.text(
        (
            x1,
            max(0, y1 - 25)
        ),
        f"SAM {i}",
        fill="blue"
    )


sam_debug.save(
    SAM_BOX_OUTPUT
)

print(
    f"Saved: {SAM_BOX_OUTPUT}"
)

display(
    sam_debug
)


# ============================================================
# STOP IF NO BOXES
# ============================================================

if len(sam_boxes) == 0:

    raise RuntimeError(
        "No valid detector boxes survived NMS. "
        "SAM cannot be run."
    )


# ============================================================
# LOAD SAM
# ============================================================

print("\n" + "=" * 60)
print("Loading SAM...")
print("=" * 60)

sam = SAM(
    SAM_MODEL_PATH
)

print(
    "SAM loaded."
)


# ============================================================
# RUN SAM WITH DETECTOR BOXES
# ============================================================

print("\n" + "=" * 60)
print("Running SAM...")
print("=" * 60)

results = sam.predict(
    source=IMAGE_PATH,
    bboxes=sam_boxes,
    verbose=False
)

result = results[0]

print(
    "SAM inference complete."
)


# ============================================================
# CHECK MASKS
# ============================================================

if result.masks is None:

    raise RuntimeError(
        "SAM returned no masks."
    )


mask_data = (
    result.masks.data
)

print(
    "\nSAM mask tensor shape:",
    mask_data.shape
)


# ============================================================
# COMBINE ALL MASKS
# ============================================================

print(
    "\nCombining masks..."
)

combined_mask = np.zeros(
    (
        original_height,
        original_width
    ),
    dtype=bool
)


for i, mask_tensor in enumerate(
    mask_data
):

    mask = (
        mask_tensor
        .cpu()
        .numpy()
    )

    # SAM normally returns masks
    # already scaled to source image.
    #
    # But we protect against a
    # different resolution here.

    if (
        mask.shape[1] != original_width
        or
        mask.shape[0] != original_height
    ):

        mask = cv2.resize(
            mask.astype(np.uint8),
            (
                original_width,
                original_height
            ),
            interpolation=cv2.INTER_NEAREST
        )

    mask_binary = (
        mask > 0
    )

    combined_mask |= (
        mask_binary
    )


# ============================================================
# CREATE ALPHA CHANNEL
# ============================================================

alpha = np.where(
    combined_mask,
    255,
    0
).astype(
    np.uint8
)


# ============================================================
# CREATE TRANSPARENT IMAGE
# ============================================================

original_rgba = np.array(
    image.convert("RGBA")
)

output = (
    original_rgba.copy()
)

output[:, :, 3] = alpha


output_image = Image.fromarray(
    output
)


# ============================================================
# SAVE TRANSPARENT PNG
# ============================================================

output_image.save(
    SAM_OUTPUT
)

print(
    f"Saved: {SAM_OUTPUT}"
)


# ============================================================
# DISPLAY TRANSPARENT RESULT
# ============================================================

print(
    "\nSAM segmentation:"
)

display(
    output_image
)


# ============================================================
# CREATE OVERLAY
# ============================================================

print(
    "\nCreating overlay..."
)

overlay = np.array(
    image.convert("RGBA")
)

# Create semi-transparent
# highlight over segmentation.

highlight = np.zeros_like(
    overlay
)

highlight[:, :, 0] = 255
highlight[:, :, 1] = 0
highlight[:, :, 2] = 0
highlight[:, :, 3] = (
    combined_mask.astype(np.uint8)
    * 100
)


base = Image.fromarray(
    overlay
)

mask_overlay = Image.fromarray(
    highlight
)

overlay_image = Image.alpha_composite(
    base,
    mask_overlay
)


# ============================================================
# DRAW SAM BOXES ON OVERLAY
# ============================================================

overlay_draw = ImageDraw.Draw(
    overlay_image
)

for i, box in enumerate(
    sam_boxes
):

    x1, y1, x2, y2 = [
        int(v)
        for v in box
    ]

    overlay_draw.rectangle(
        [
            x1,
            y1,
            x2,
            y2
        ],
        outline="blue",
        width=4
    )

    overlay_draw.text(
        (
            x1,
            max(0, y1 - 20)
        ),
        f"SAM {i}",
        fill="blue"
    )


# ============================================================
# SAVE OVERLAY
# ============================================================

overlay_image.save(
    OVERLAY_OUTPUT
)

print(
    f"Saved: {OVERLAY_OUTPUT}"
)


# ============================================================
# DISPLAY OVERLAY
# ============================================================

print(
    "\nSAM overlay:"
)

display(
    overlay_image
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(
    f"Image:              {IMAGE_PATH}"
)

print(
    f"Image size:         "
    f"{original_width} x {original_height}"
)

print(
    f"Core ML input:      "
    f"{MODEL_SIZE} x {MODEL_SIZE}"
)

print(
    f"Confidence threshold: "
    f"{CONF_THRESHOLD}"
)

print(
    f"IoU threshold:       "
    f"{IOU_THRESHOLD}"
)

print(
    f"Detected objects:    "
    f"{len(boxes)}"
)

print(
    f"SAM masks:           "
    f"{len(mask_data)}"
)

print("\nOutput files:")

print(
    f"  {DETECTOR_OUTPUT}"
)

print(
    f"  {SAM_BOX_OUTPUT}"
)

print(
    f"  {SAM_OUTPUT}"
)

print(
    f"  {OVERLAY_OUTPUT}"
)

print("=" * 60)